# 長い質問で最初の応答とその後の生成がどれだけ遅くなるか測る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_llm/llm_prefill_decode.ipynb)

NLP クックブック 14-15 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

RAG で参考文書を増やしたときの待ち時間を調べたい。小型 LLM に長さだけを変えた入力を渡し、最初の1トークンを返すまでと、続く31トークンを生成する区間を分けて測る。

In [1]:
import time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Colab のランタイムを GPU に変更してください")
print(torch.cuda.get_device_name())
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).cuda().eval()
context = tok.encode("この図書館は午前九時に開館します。", add_special_tokens=False)
question = tok.encode("質問：この図書館の開館時刻は？", add_special_tokens=False)

def make_input(length):
    n = length - len(question)
    prefix = (context * (n // len(context) + 1))[:n]
    return torch.tensor([prefix + question], device="cuda")

Tesla T4


In [2]:
@torch.inference_mode()
def measure(ids, cached):
    # 入力はあらかじめ GPU に置き、トークン化・転送を計測から外す。
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    out = model(ids, use_cache=cached, logits_to_keep=1)
    token = out.logits[:, -1].argmax(-1, keepdim=True)
    torch.cuda.synchronize()
    first = time.perf_counter()
    for _ in range(31):
        if cached:
            out = model(token, past_key_values=out.past_key_values, use_cache=True, logits_to_keep=1)
        else:
            ids = torch.cat([ids, token], dim=1)
            out = model(ids, use_cache=False, logits_to_keep=1)
        token = out.logits[:, -1].argmax(-1, keepdim=True)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - first
    return {"初回 ms": (first - start) * 1000, "続き token/s": 31 / elapsed,
            "ピーク MiB": torch.cuda.max_memory_allocated() / 2**20}

rows = []
for length, cached in [(128, True), (512, True), (2048, True), (128, False)]:
    ids = make_input(length)
    measure(ids, cached)  # 同じ条件でウォームアップ
    runs = pd.DataFrame([measure(ids, cached) for _ in range(3)])
    rows.append({"入力トークン": length, "KV キャッシュ": cached, **runs.median().to_dict()})
print(pd.DataFrame(rows).round(2).to_string(index=False))

 入力トークン  KV キャッシュ  初回 ms  続き token/s  ピーク MiB
    128      True  25.18       44.43   965.26
    512      True  30.87       34.69   982.17
   2048      True  99.87       44.08  1056.82
    128     False  23.15       42.34   964.56


- 入力には同じ参考文を繰り返し、末尾に同じ質問を置く。貪欲法で文末記号が出ても32トークンまで続け、出力長を固定する。表は3回の中央値。
- キャッシュありの3行では、質問全体を処理する最初の待ち時間と、その後の生成速度を別々に読む。ピークはモデルの重みを含む PyTorch の割り当て量。
- 128トークンの2行では、過去のキーと値を GPU に残して次の1トークンだけを入力する場合と、毎回全文を入力し直す場合の時間とメモリを比べる。